# Sosa, Plitt & Giocomo (2025) on dandiset 001361

Paper. M. Sosa, M. H. Plitt, L. M. Giocomo, "A flexible hippocampal population code for experience relative to reward", *Nature Neuroscience* 28, 1497-1509 (2025), [doi:10.1038/s41593-025-01985-4](https://doi.org/10.1038/s41593-025-01985-4).

Dataset. [DANDI:001361](https://dandiarchive.org/dandiset/001361/0.251124.0550), version 0.251124.0550 (DOI 10.48324/dandi.001361/0.251124.0550): 152 NWB files, 11 switch mice (m3, m4, m7, m11 to m15, m17 to m19), 14 days each (m11 from day 3), two-photon imaging of CA1 with behaviour on the imaging frames.

Why this version. The paper cites version 0.250406.0045, and the protocol says to pin the cited version. In that version (and in 0.250609.2249, whose total size is the same) every file of a mouse carries the behaviour of one session: there are 11 distinct behaviour series for 152 files, and the behaviour matches the length of the imaging in only 11 files (the first cell below shows this from the files). The neural data of 141 files therefore cannot be aligned with the behaviour of their own session, and no claim can be tested there. Version 0.251124.0550, the latest, has 152 distinct behaviour series whose length equals that of the imaging (to one frame in 10 files), and its zones agree with the scene names; it is pinned here. This is recorded as a deviation on every claim and as a finding.

Text the claims were written from. The version of record (Europe PMC, PMC12229899) with Methods, Extended Data legends and the Reporting Summary; the paper's code, github.com/GiocomoLab/Sosa_et_al_2024 at commit adf0f9e. The claims file is `claims/001361/10.1038-s41593-025-01985-4.yaml`.

What the analysis computes from the files. dF/F and the deconvolved activity are computed from the files' raw fluorescence and neuropil as the Methods and the code give them (trial-wise maximin baseline, OASIS on dF/F); the files' own "Deconvolved" series is not the paper's deconvolved activity (data check) and is used only as a variant. Every random step has its own seeded generator.

| Claim | Tested here |
|---|---|
| fig1h-lick-ratio-improves | tested |
| place-cell-fraction | tested |
| remap-category-fractions | tested |
| ed2cd-switch-remapping-intermediate | tested |
| ed2f-teleport-distance | tested |
| fig2e-rr-remapping-above-chance | tested |
| fig2h-rr-remapping-grows | tested |
| fig2eg-mean-rr-position-after-reward | tested |
| fig2i-rr-remapping-spans-track | tested |
| ed2jl-rr-remapping-beyond-50cm | tested |
| ed2m-more-fields-after-reward | tested |
| ed2n-far-remapping-no-increase | tested |
| rr-cell-fractions | tested |
| ed2pq-rr-reward-aligned-correlation | tested |
| fig3b-decode-rr-position | blocked (too large) |
| fig3c-decode-range | blocked (too large) |
| fig4c-rr-sequences-preserved | tested |
| fig4f-tr-sequences-preserved | tested |
| fig4i-nonrr-sequences-not-preserved | tested |
| ed5a-tr-fewer-across-environments | tested |
| ed5f-rr-no-decrease-across-environments | tested |
| fig4j-rr-overrepresent-reward | tested |
| fig4m-rr-density-near-reward-grows | tested |
| fig4no-licking-speed-increase | tested |
| fig4p-lick-precision-sequence-precision | tested |
| ed4hij-sequence-variance-decreases | tested |
| fig4qr-tr-track-ends | tested |
| ed4l-disappearing-decrease | tested |
| ed5-teleport-fractions | blocked (not attempted in the run's time) |
| fig5b, fig5d, fig5f, fig5h, fig5i, fig5jkl | tested (cells matched across days here) |
| ed6a-rr-near-reward-stable, ed6c-rr-shift-toward-reward, ed6-across-day-drift | tested |
| ed7-no-anatomical-bias | tested (depth part: no data) |
| fig6e-rr-prefer-reward, ed8k-rr-prefer-reward-after-switch, ed8df-time-warp-speed-mse | tested |
| glm-well-fit-cells, fig7de-glm-top-predictors | blocked (too large) |
| fig8e-rr-remap-precedes-behavior, ed10-appearing-remap-later | blocked (not attempted in the run's time) |
| ed3d, ed3fg, ed3mo, ed3p (individual fields) | tested |

The notebook takes about 35 minutes, most of it streaming the 152 files.

In [ ]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import sosa
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

## Data check

The first cell compares the behaviour of every file in the cited and the pinned versions. The next ones process all 152 sessions (switch days with the RR cross-correlation), then print counts by subject, session and condition, the ranges of the variables used, and two checks that the imaging and the behaviour share a clock.

In [ ]:
# Versions: the paper cites 0.250406.0045; this entry reads 0.251124.0550 (see the header).
# The behaviour of every file in both versions is compared here (position series only, small).
import hashlib
def beh_digest(version):
    out = {}
    for r in sosa.session_table(version):
        f = sosa.open_asset(r["asset_id"])
        pos = f[sosa.B + "position/data"][:]
        out[(r["mouse"], r["day"])] = (hashlib.md5(pos.tobytes()).hexdigest(), len(pos),
                                       f[sosa.O + "Deconvolved/plane0/data"].shape[0])
        f.close()
    return out
dig_cited = beh_digest(sosa.CITED_VERSION)
dig_used = beh_digest(sosa.VERSION)
vt = pd.DataFrame([dict(mouse=k[0], day=k[1], cited_md5=v[0], cited_n_beh=v[1], cited_n_img=v[2],
                        used_md5=dig_used[k][0], used_n_beh=dig_used[k][1], used_n_img=dig_used[k][2])
                   for k, v in dig_cited.items()])
print("files:", len(vt))
print("cited version: distinct behaviour series", vt.cited_md5.nunique(),
      "; files whose behaviour length equals the imaging length", (vt.cited_n_beh == vt.cited_n_img).sum())
print("distinct behaviour series per mouse, cited version:", vt.groupby("mouse").cited_md5.nunique().to_dict())
print("used version: distinct behaviour series", vt.used_md5.nunique(),
      "; behaviour length = imaging length", (vt.used_n_beh == vt.used_n_img).sum(),
      "; differ by one frame", (abs(vt.used_n_beh - vt.used_n_img) == 1).sum(),
      "; differ by more", (abs(vt.used_n_beh - vt.used_n_img) > 1).sum())

In [ ]:
# Process every session (all 152 files): behaviour per trial, dF/F, deconvolved events, spatial
# information with its shuffle, peaks, circular maps and the RR cross-correlation on switch days.
# About 30 minutes, most of it streaming the files.
import time
rows = sosa.session_table()
t0 = time.time()
res = sosa.run_all(rows, workers=4)
print(f"{len(res)} sessions processed in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Data check: counts, scenes, zones, trials, cells, value ranges
tab = []
for (m, d), r in res.items():
    s0, s1 = r["sets"]
    fz = r["flag_zone"]
    a0 = np.mean([z == r["zones"][0] for z in fz[s0] if z])
    a1 = np.mean([z == r["zones"][1] for z in fz[s1] if z])
    tab.append(dict(mouse=m, day=d, scene=r["scene"], switch=r["is_switch"], n_trials=r["n_trials"],
                    n_img=r["n_img"], n_beh=r["n_beh"], rois=r["n_rois_total"], cells=r["n_cells"],
                    flag_zone_agrees_pre=a0, flag_zone_agrees_post=a1,
                    rewarded=r["beh"]["rewarded"].mean(), sensor_err=r["beh"]["sensor_err"].sum(),
                    interneurons=r["is_int"].sum()))
tab = pd.DataFrame(tab)
print("mice:", sorted(tab.mouse.unique(), key=lambda s: int(s[1:])), "n =", tab.mouse.nunique())
print("sessions per mouse:", tab.groupby("mouse").size().to_dict())
print("days per mouse:", {m: sorted(g.day) for m, g in tab.groupby("mouse")})
print("switch days found from the scene names:", sorted(tab[tab.switch].day.unique()))
print("switch sessions:", tab.switch.sum(), "; stay sessions:", (~tab.switch).sum())
print("trials per session: mean %.1f s.d. %.1f, range %d to %d (paper 80.5 +/- 7.4)" % (
    tab.n_trials.mean(), tab.n_trials.std(), tab.n_trials.min(), tab.n_trials.max()))
sw = tab[tab.switch]
print("cells (iscell) per switch session: mean %.0f s.d. %.0f; all sessions range %d to %d (paper 155 to 2,172)" % (
    sw.cells.mean(), sw.cells.std(), tab.cells.min(), tab.cells.max()))
print("rewarded fraction of trials: mean %.3f (omissions %.3f; paper ~15%% omitted, plus misses)" % (
    tab.rewarded.mean(), 1 - tab.rewarded.mean()))
print("zone named by the scene agrees with the reward_zone flag in rewarded trials: pre %.3f, post %.3f (min %.3f)" % (
    tab.flag_zone_agrees_pre.mean(), tab.flag_zone_agrees_post.mean(),
    min(tab.flag_zone_agrees_pre.min(), tab.flag_zone_agrees_post.min())))
print("trials with lick-sensor errors: %d of %d (paper 81 of 12,376 on all imaged trials)" % (
    tab.sensor_err.sum(), tab.n_trials.sum()))
print("putative interneurons (dF/F-speed r > 0.5): %.2f%% of cells (paper 0.42 +/- 0.85%%)" % (
    100 * (tab.interneurons / tab.cells).mean()))
print("environments by day 8 scene:", tab[tab.day == 8].set_index("mouse").scene.to_dict())
r0 = next(iter(res.values()))
print("dF/F 1st-99th percentile, deconvolved events min and 99th percentile, per session (ranges over sessions):")
print("  dF/F low %.3f..%.3f high %.3f..%.3f" % (
    min(r["dff_range"][0] for r in res.values()), max(r["dff_range"][0] for r in res.values()),
    min(r["dff_range"][1] for r in res.values()), max(r["dff_range"][1] for r in res.values())))
print("  events min %.3f, 99th pct %.3f..%.3f (dF/F units)" % (
    min(r["dec_range"][0] for r in res.values()), min(r["dec_range"][1] for r in res.values()),
    max(r["dec_range"][1] for r in res.values())))
print("  dataset 'Deconvolved' 99th pct %.1f..%.1f ('lumens', the unit the file gives)" % (
    min(r["ds_dec_range"][1] for r in res.values()), max(r["ds_dec_range"][1] for r in res.values())))
print("median correlation of the dataset's 'Deconvolved' with OASIS events on dF/F, per session: median %.2f, range %.2f..%.2f" % (
    np.median([r["r_events_vs_ds"] for r in res.values()]),
    min(r["r_events_vs_ds"] for r in res.values()), max(r["r_events_vs_ds"] for r in res.values())))
pos_lo = min(np.nanmin(r["beh"]["speed2"]) for r in res.values())
print("binned speed range over sessions: %.1f to %.1f cm/s" % (
    pos_lo, max(np.nanmax(r["beh"]["speed2"]) for r in res.values())))

In [ ]:
# Clock checks. (1) Where the laser was lowered in the teleport, dark imaging frames should fall on
# the behaviour's teleport frames at lag 0. (2) In every session, place maps from odd and even
# trials agree when activity and position are aligned and not when shifted by 300 frames (~20 s).
ck = []
for (m, d), r in res.items():
    c = r["clock"]
    ck.append(dict(mouse=m, day=d, dark_frac=c["dark_frac"], best_lag=c.get("best_lag", np.nan),
                   agree0=c.get("agree0", np.nan), agree_far=c.get("agree_far", np.nan),
                   **{f"r_lag{k}": v for k, v in c["split_half"].items()}))
ck = pd.DataFrame(ck)
dk = ck[ck.dark_frac >= 0.02]
print("sessions with dark teleport frames: %d of %d" % (len(dk), len(ck)))
print("best lag (frames) distribution:", dk.best_lag.value_counts().sort_index().to_dict())
print("agreement of dark and teleport frames: at lag 0 median %.3f (min %.3f); at lag +/-60 median %.3f" % (
    dk.agree0.median(), dk.agree0.min(), dk.agree_far.median()))
lagcols = [c for c in ck.columns if c.startswith("r_lag")]
print("median odd/even map correlation over cells, by shift of activity (frames); median and min over sessions:")
print(ck[lagcols].agg(["median", "min"]).round(3).to_string())
print("sessions where the aligned correlation exceeds both 300-frame shifts:",
      ((ck["r_lag0"] > ck["r_lag300"]) & (ck["r_lag0"] > ck["r_lag-300"])).sum(), "of", len(ck))
print("sessions with no dark frames (imaged through the teleport):",
      ck[ck.dark_frac < 0.02][["mouse", "day"]].values.tolist())

In [ ]:
# Cell classes per switch session (Methods, Remapping category definitions; Criteria to define RR cells)
def classes(r, key="m", lagkey="lag_meth", exclude_rr="methods", sig="sig"):
    pc0, pc1 = r[sig]
    ii = r["is_int"]
    both = pc0 & pc1 & ~ii
    por = (pc0 | pc1) & ~ii
    p0, p1 = r["peak_lin"]
    z0, z1 = r["zone_starts"]
    near = both & (np.abs(p0 - z0) <= 50) & (np.abs(p1 - z1) <= 50)
    tr = both & (np.abs(p0 - p1) <= 50) & ~near
    far = both & ~tr & ~near
    tm0, tm1 = r["trial_mean"]
    m0 = np.nanmean(tm0, 0); m1 = np.nanmean(tm1, 0); sd0 = np.nanstd(tm0, 0)
    q50 = np.nanpercentile(tm0, 50, axis=0)
    app = ~pc0 & pc1 & ~ii & (m1 > m0 + sd0)
    dis = pc0 & ~pc1 & ~ii & (m1 < q50)
    c = r["circ"][key]
    rr = c["cand"] & (np.abs(c[lagkey]) <= 5) if r["is_switch"] else np.zeros_like(por)
    if exclude_rr in ("methods", "code"):
        app = app & ~rr; dis = dis & ~rr
    if exclude_rr == "code":
        tr = tr & ~rr; near = near & ~rr; far = far & ~rr
    nonrr = far & ~rr
    uncl = por & ~(rr | tr | dis | app | near | far)
    nonpc = ~(pc0 | pc1) & ~ii
    return dict(pc=por, both=both, tr=tr, near=near, far=far, app=app, dis=dis, rr=rr, nonrr=nonrr,
                uncl=uncl, nonpc=nonpc)

SW = sosa.SWITCH_DAYS
mice = sorted({m for m, d in res}, key=lambda s: int(s[1:]))
CL = {k: classes(r) for k, r in res.items() if k[1] in SW}
CLc = {k: classes(r, key="c", lagkey="lag_code", exclude_rr="code") for k, r in res.items() if k[1] in SW}
def frac_table(CLd, cats=("tr", "dis", "app", "near", "far", "rr", "nonrr", "uncl")):
    rows = []
    for (m, d), C in CLd.items():
        n = C["pc"].sum()
        rows.append(dict(mouse=m, day=d, cells=res[(m, d)]["n_cells"], pc=n,
                         **{c: 100 * C[c].sum() / n for c in cats}, **{"n_" + c: C[c].sum() for c in cats}))
    return pd.DataFrame(rows).sort_values(["mouse", "day"])
FT = frac_table(CL)
FTc = frac_table(CLc)
print(FT.groupby("day")[["pc", "tr", "dis", "app", "near", "far", "rr", "nonrr", "uncl"]].mean().round(1))

In [ ]:
def mixed(df, y, x="switch", logit=False):
    """Linear mixed-effects model y ~ 1 + x, random intercept per mouse, REML, Wald test
    (Methods, Linear mixed-effects models; statsmodels mixedlm)."""
    d = df[["mouse", x, y]].dropna().copy()
    if logit:
        d = d[(d[y] > 0) & (d[y] < 1)]
        d[y] = np.log(d[y] / (1 - d[y]))
    fit = smf.mixedlm(f"{y} ~ {x}", d, groups=d["mouse"]).fit(reml=True)
    return fit.params[x], fit.pvalues[x], len(d)

def holm(p):
    return multipletests(p, method="holm")[1]

def friedman_posthoc(wide):
    """Friedman over columns (days), then pairwise two-sided Wilcoxon signed-rank, Holm over all pairs."""
    w = wide.dropna()
    q, p = st.friedmanchisquare(*[w[c] for c in w.columns])
    pairs, ps = [], []
    cols = list(w.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append((cols[i], cols[j])); ps.append(st.wilcoxon(w[cols[i]], w[cols[j]]).pvalue)
    return q, p, len(w), pd.DataFrame(dict(a=[a for a, b in pairs], b=[b for a, b in pairs], p=ps, p_holm=holm(ps)))

SWI = {d: i + 1 for i, d in enumerate(sosa.SWITCH_DAYS)}     # switch number 1..7

## fig1h-lick-ratio-improves

> After each reward switch, every switch mouse improved its anticipatory licking before the new reward zone and kept that precise licking, as measured by the anticipatory lick ratio over blocks of ten trials.

In [ ]:
def lick_ratio_trials(r, zone_start):
    b = r["beh"]
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = b["lick_cnt"] / np.where(b["occ_t"] > 0, b["occ_t"], np.nan)
    centers = 5 + 10 * np.arange(45)
    zs = np.asarray(zone_start)[:, None]
    ins = (centers[None, :] >= zs - 50) & (centers[None, :] < zs)
    outs = ~((centers[None, :] >= zs - 50) & (centers[None, :] < zs + 50))
    lin = np.nanmean(np.where(ins, rate, np.nan), 1)
    lout = np.nanmean(np.where(outs, rate, np.nan), 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        ratio = (lin - lout) / (lin + lout)
    ratio[b["sensor_err"]] = np.nan
    return ratio

def blocks(x):
    n = len(x)
    return [np.nanmean(c) if np.any(np.isfinite(c)) else np.nan for c in np.array_split(x, int(np.ceil(n / 10)))] if n else []

LR = []
for (m, d), r in res.items():
    nt = r["n_trials"]
    zst = np.zeros(nt)
    for k, s in enumerate(r["sets"]):
        zst[s] = r["zone_starts"][k]
    ratio = lick_ratio_trials(r, zst)
    s0, s1 = r["sets"]
    if r["is_switch"]:
        b_pre, b_post = blocks(ratio[s0]), blocks(ratio[s1])
    else:
        b_pre, b_post = blocks(ratio), []
    LR.append(dict(mouse=m, day=d, switch=r["is_switch"], first_block_day=b_pre[0] if b_pre else np.nan,
                   first_post=b_post[0] if b_post else np.nan, last_post=b_post[-1] if b_post else np.nan,
                   n_post_blocks=len(b_post)))
LR = pd.DataFrame(LR).sort_values(["mouse", "day"])
sw = LR[LR.switch].copy()
sw["improve"] = sw.last_post - sw.first_post
# retention: first block of the next day (stay day, or next switch day's pre-switch trials)
nxt = LR.set_index(["mouse", "day"]).first_block_day
sw["next_day_first_block"] = [nxt.get((m, d + 1), np.nan) for m, d in zip(sw.mouse, sw.day)]
per_mouse = sw.groupby("mouse").agg(days_improved=("improve", lambda x: int((x > 0).sum())),
                                     n_days=("improve", "size"), mean_improve=("improve", "mean"),
                                     mean_next_day=("next_day_first_block", "mean"),
                                     n_next=("next_day_first_block", lambda x: int(x.notna().sum())))
print(per_mouse.round(3))
a_mice = int((per_mouse.days_improved == per_mouse.n_days).sum())
b_mice = int((per_mouse.mean_improve > 0).sum())
ret = int((per_mouse.mean_next_day > 0).sum())
print(f"reading (a): mice improving on every switch day {a_mice} of {len(per_mouse)}; mouse-days improving {int((sw.improve > 0).sum())} of {len(sw)}")
print(f"reading (b): mice whose mean (last - first post block) > 0: {b_mice} of {len(per_mouse)}")
print(f"retention: mice whose mean first-block ratio on the next day > 0: {ret} of {len(per_mouse)}")
print("first and last post-switch block, mean over mouse-days: %.3f, %.3f" % (sw.first_post.mean(), sw.last_post.mean()))

**Result: reproduced under reading (b).** Every mouse's mean change from the first to the last post-switch block is positive (11 of 11), and every mouse's mean ratio in the first block of the following day is above 0 (11 of 11). Under reading (a), 8 of 11 mice improve on every switch day (74 of 77 mouse-days).

## place-cell-fraction

> On switch days, 459 +/- 263 of 954 +/- 453 imaged cells per session were place cells, 48.5 +/- 14.5% (mean +/- s.d. across 11 mice and seven switch sessions).

In [ ]:
FT["frac_pc"] = 100 * FT.pc / FT.cells
print("over the 77 mouse and switch-day sessions:")
print("  place cells %.1f +/- %.1f; cells %.1f +/- %.1f; %% place cells %.1f +/- %.1f" % (
    FT.pc.mean(), FT.pc.std(), FT.cells.mean(), FT.cells.std(), FT.frac_pc.mean(), FT.frac_pc.std()))
pm = FT.groupby("mouse")[["pc", "cells", "frac_pc"]].mean()
print("variant, mean over mice of per-mouse means: place cells %.1f, cells %.1f, %% %.1f" % (
    pm.pc.mean(), pm.cells.mean(), pm.frac_pc.mean()))
print("per mouse:"); print(pm.round(1))
print("per day:"); print(FT.groupby("day")[["pc", "cells", "frac_pc"]].mean().round(1))
# variant: spatial information on the dataset's own 'Deconvolved' series
FTd = frac_table({k: classes(res[k], sig="sig_ds") for k in CL}, cats=("tr",))
FTd["frac_pc"] = 100 * FTd.pc / FTd.cells
print("variant, SI on the dataset's 'Deconvolved' series: place cells %.1f +/- %.1f, %.1f +/- %.1f%%" % (
    FTd.pc.mean(), FTd.pc.std(), FTd.frac_pc.mean(), FTd.frac_pc.std()))
pc_before = np.mean([res[k]["sig"][0][~res[k]["is_int"]].mean() for k in CL])
pc_after = np.mean([res[k]["sig"][1][~res[k]["is_int"]].mean() for k in CL])
print("fraction of cells significant before / after the switch, mean over sessions: %.3f / %.3f" % (pc_before, pc_after))

**Result: partially reproduced.** The cells imaged per switch session match (970 +/- 482 against 954 +/- 95 tolerance), but place cells are 577 +/- 331 (expected 459 +/- 45.9) and 60.0 +/- 16.1% of cells (expected 48.5 +/- 4.85). The variant over per-mouse means gives the same values, and the files' own deconvolved series gives more place cells (66.8%).

## remap-category-fractions

> After a switch within an environment, 21.4 +/- 7.9% of place cells kept a field at the same track position (TR), 11.2 +/- 7.0% lost their field (disappearing), 6.8 +/- 4.0% gained one (appearing), 4.8 +/- 3.3% followed the reward within 50 cm of both zone starts (remap near reward) and 15.6 +/- 3.6% remapped with fields far from reward (remap far from reward).

In [ ]:
exp = dict(tr=21.4, dis=11.2, app=6.8, near=4.8, far=15.6)
for name, T in [("(a) Methods: RR removed from appearing and disappearing only", FT),
                ("(b) code: RR removed from every category, code's 2pi/44 bins", FTc)]:
    w = T[T.day != 8]
    print(name, f"; {len(w)} mouse-days")
    for c, v in exp.items():
        mu, sd = w[c].mean(), w[c].std()
        print(f"   {c:5s} {mu:5.1f} +/- {sd:4.1f}  expected {v} +/- {0.1 * v:.2f}  within: {abs(mu - v) <= 0.1 * v + 1e-9}")

**Result: partially reproduced.** Under the Methods' reading TR (22.7%) and remap near reward (4.8%) are within tolerance; disappearing (8.6% against 11.2), appearing (4.9% against 6.8) and remap far (17.9% against 15.6) are not. Under the code's reading only TR is within tolerance.

## rr-cell-fractions

> Under the additional criteria, RR cells were 16.3 +/- 5.3% of place cells over all switch days, 13.7 +/- 3.9% on the first switch and 19.4 +/- 5.2% on the last; non-RR remapping cells were 11.9 +/- 3.0%, and 31.4 +/- 14.3% of place cells fell in no category.

In [ ]:
def rr_report(T, label):
    w6 = T[T.day != 8]
    vals = [("RR all switch days", T.rr, 16.3), ("RR day 3", T[T.day == 3].rr, 13.7),
            ("RR day 14", T[T.day == 14].rr, 19.4)]
    print(label)
    for nm, x, v in vals:
        print(f"   {nm:22s} {x.mean():5.1f} +/- {x.std():4.1f}  expected {v}  within: {abs(x.mean() - v) <= 0.1 * v}")
    for rd, w in [("(a) all 7 days", T), ("(b) 6 within-env days", w6)]:
        print(f"   non-RR {rd}: {w.nonrr.mean():.1f} +/- {w.nonrr.std():.1f} (exp 11.9, within {abs(w.nonrr.mean() - 11.9) <= 1.19}); "
              f"unclassified {w.uncl.mean():.1f} +/- {w.uncl.std():.1f} (exp 31.4, within {abs(w.uncl.mean() - 31.4) <= 3.14})")
rr_report(FT, "Methods (2pi/45 bins, shuffle 1-45 bins)")
rr_report(FTc, "code variant (2pi/44 bins, shuffle 1-44 bins)")
print("RR % of place cells by mouse and day:"); print(FT.pivot(index="mouse", columns="day", values="rr").round(1))
print("pooled counts: RR %d, TR %d, non-RR remapping %d" % (FT.n_rr.sum(), FT.n_tr.sum(), FT.n_nonrr.sum()))

**Result: partially reproduced.** With the Methods' bins and shuffle, RR cells on day 3 (12.8%) and unclassified cells (34.4%) are within tolerance, while RR over all days (14.6%, limit 14.67), RR on day 14 (17.1%, limit 17.46) and non-RR remapping (14.3%, expected 11.9 +/- 1.19) are not. With the code's bins the three RR values are within tolerance (14.7, 12.8, 17.5%), and non-RR remapping still is not (14.3 and 13.9%).

## ed5a-tr-fewer-across-environments

> The environment change reduced the proportion of TR cells, 8.4 +/- 3.1% of place cells on day 8 against 21.4 +/- 7.9% on switches within an environment.

In [ ]:
FT["tr_norr"] = 100 * np.array([(CL[(m, d)]["tr"] & ~CL[(m, d)]["rr"]).sum() for m, d in zip(FT.mouse, FT.day)]) / FT.pc
W = FT.pivot(index="mouse", columns="day", values="tr_norr")
q, p, n, ph = friedman_posthoc(W)
print(W.round(1))
print(f"Friedman Q = {q:.1f}, p = {p:.2g}, n = {n} mice (paper Q = 32.4, p = 1.4e-5)")
d8 = ph[(ph.a == 8) | (ph.b == 8)].copy()
d8["day8_lower"] = [W[8].median() < W[b if a == 8 else a].median() for a, b in zip(d8.a, d8.b)]
print(d8.round(4).to_string(index=False))
v8 = FT[FT.day == 8].tr_norr; vw = FT[FT.day != 8].tr_norr
print("TR day 8: %.1f +/- %.1f (exp 8.4 +/- 0.84); within-env pooled 66 mouse-days: %.1f +/- %.1f (exp 21.4 +/- 2.14)" % (
    v8.mean(), v8.std(), vw.mean(), vw.std()))
print("day 8 lower than every other day at Holm p < 0.05:", bool((d8.p_holm < 0.05).all() and d8.day8_lower.all()))

**Result: partially reproduced.** The Friedman test finds an effect of day (Q = 26.8, p = 0.00016), and day 8 has the lowest TR fraction, but day 8 against day 10 is not significant after Holm correction (p = 0.073). The within-environment value (22.7%) is within tolerance; day 8 (10.5%) is not (8.4 +/- 0.84).

## ed5f-rr-no-decrease-across-environments

> The proportion of RR cells did not decrease upon the environment switch on day 8.

In [ ]:
W = FT.pivot(index="mouse", columns="day", values="rr")
q, p, n, ph = friedman_posthoc(W)
print(f"Friedman Q = {q:.1f}, p = {p:.3g}, n = {n}")
d8 = ph[(ph.a == 8) | (ph.b == 8)].copy()
d8["day8_median"] = W[8].median()
d8["other_median"] = [W[b if a == 8 else a].median() for a, b in zip(d8.a, d8.b)]
print(d8.round(4).to_string(index=False))
print("any post hoc with day 8 lower at Holm p < 0.05:", bool(((d8.p_holm < 0.05) & (d8.day8_median < d8.other_median)).any()))
print("medians by day:", W.median().round(1).to_dict())

**Result: reproduced.** No post hoc comparison shows day 8 lower than another day (day 8 median 15.2%, all Holm p >= 0.26; Friedman Q = 12.8, p = 0.046).

## ed2pq-rr-reward-aligned-correlation

> RR cells had higher pre- against post-switch correlations of their reward-aligned activity maps than TR and non-RR remapping cells, and TR cells had higher correlations on the original track axis than RR and non-RR cells.

In [ ]:
def pool(cat, key):
    return np.concatenate([res[k]["circ"]["m"][key][CL[k][cat]] for k in CL])
rr_rew, tr_rew, nr_rew = pool("rr", "r_rew"), pool("tr", "r_rew"), pool("nonrr", "r_rew")
rr_tr, tr_tr, nr_tr = pool("rr", "r_track"), pool("tr", "r_track"), pool("nonrr", "r_track")
print("n: RR %d, TR %d, non-RR %d (expected 5979, 7027, 4314 +/- 10%%)" % (len(rr_rew), len(tr_rew), len(nr_rew)))
tests = [("reward-aligned RR > TR", rr_rew, tr_rew), ("reward-aligned RR > non-RR", rr_rew, nr_rew),
         ("track TR > RR", tr_tr, rr_tr), ("track TR > non-RR", tr_tr, nr_tr)]
for nm, a, b in tests:
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    z, p = st.ranksums(a, b)
    print(f"{nm:28s} medians {np.median(a):.3f} vs {np.median(b):.3f}, Z = {z:.1f}, p = {p:.2g}")

**Result: reproduced.** All four rank-sum tests go in the stated direction with p far below 0.001 (Z = 100.1, 82.3, 102.9 and 87.0). The pooled counts (RR 6,762, TR 9,442, non-RR 6,417) are above the paper's.

In [ ]:
# RR remapping against chance (Methods, Quantification of RR remapping compared to chance;
# code, dayData.add_circ_relative_peaks and _frac_hist_above_shuf)
RAD50 = sosa.RAD50
CM = sosa.TRACK / (2 * np.pi)          # cm per radian
def remap_cells(k, key="m"):
    """Cells with significant SI before and after (interneurons out), TR cells (track peaks within
    0.698 rad) removed. Returns rel0, rel1, post peak bin, zone-1 start in rad, centers."""
    r = res[k]; c = r["circ"][key]
    pc0, pc1 = r["sig"]
    both = pc0 & pc1 & ~r["is_int"]
    cen = c["centers"]
    trk = np.abs(sosa.wrap(cen[c["peak1"]] - cen[c["peak0"]])) <= RAD50 + 1e-9
    keep = both & ~trk & (c["peak0"] >= 0) & (c["peak1"] >= 0)
    return dict(rel0=c["rel0"][keep], rel1=c["rel1"][keep], pk1=c["peak1"][keep],
                rz1=sosa.cm_to_rad(r["zone_starts"][1]), centers=cen, mouse=k[0])

def above_chance(cells, rng, nperm=1000, window=RAD50, excl=None):
    """Histogram of the circular difference of relative peaks (after minus before) in 45 bins
    of 2pi/45, as a fraction of cells, against a shuffle that rolls each post-switch map by
    0-44 bins (the peak moves with the map). Returns the summed excess over the shuffle's 95th
    percentile in bins within `window` of 0, the histogram, the 95th percentile and n."""
    rel0 = np.concatenate([c["rel0"] for c in cells]); rel1 = np.concatenate([c["rel1"] for c in cells])
    pk1 = np.concatenate([c["pk1"] for c in cells]); rz1 = np.concatenate([np.full(len(c["pk1"]), c["rz1"]) for c in cells])
    cen = cells[0]["centers"]
    if excl is not None:   # remove cells with peaks within excl rad of both zone starts
        ok = ~((np.abs(rel0) <= excl) & (np.abs(rel1) <= excl))
        rel0, rel1, pk1, rz1 = rel0[ok], rel1[ok], pk1[ok], rz1[ok]
    n = len(rel0)
    edges = -np.pi + 2 * np.pi / 45 * np.arange(46)
    bc = edges[:-1] + np.pi / 45
    h = np.histogram(sosa.wrap(rel1 - rel0), edges)[0] / max(n, 1)
    nb = len(cen)
    hn = np.zeros((nperm, 45))
    for p in range(nperm):
        k = rng.integers(0, nb, size=n)
        nul = sosa.wrap(cen[(pk1 + k) % nb] - rz1)
        hn[p] = np.histogram(sosa.wrap(nul - rel0), edges)[0] / max(n, 1)
    q95 = np.percentile(hn, 95, axis=0)
    inc = np.abs(bc) <= window
    ex = h[inc] - q95[inc]
    return ex[ex > 0].sum(), h, q95, n, bc, rel0, rel1

REM = {k: remap_cells(k) for k in CL}
REMc = {k: remap_cells(k, key="c") for k in CL}

## fig2e-rr-remapping-above-chance

> On the first switch day the fraction of remapping place cells whose firing kept the same position relative to reward exceeded a random-remapping shuffle.

In [ ]:
rng = np.random.default_rng(2001)
AC = {}
for d in sosa.SWITCH_DAYS:
    AC[d] = above_chance([REM[(m, d)] for m in mice], rng)
    AC[d + 100] = above_chance([REMc[(m, d)] for m in mice], rng)
for d in sosa.SWITCH_DAYS:
    fr, h, q95, n, bc, _, _ = AC[d]
    inc = np.abs(bc) <= RAD50
    print(f"day {d:2d}: remapping cells {n:5d}; bins near 0 above the shuffle's 95th pct {int((h[inc] > q95[inc]).sum())} of {inc.sum()}; "
          f"above-chance fraction {fr:.3f}; code bins variant {AC[d + 100][0]:.3f} (n {AC[d + 100][3]})")
print("expected remapping cells: day 3 1082 +/- 108, day 14 1153 +/- 115")
fr, h, q95, n, bc, _, _ = AC[3]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(bc, h, width=2 * np.pi / 45, color="0.6", label="data")
ax.step(bc, q95, where="mid", color="k", label="shuffle 95th pct")
ax.set_xlabel("relative peak after minus before (rad)"); ax.set_ylabel("fraction of cells"); ax.legend(); ax.set_title("day 3")
plt.show()
print("plotted range: fraction %.3f to %.3f, all bins shown" % (h.min(), h.max()))

**Result: reproduced.** On day 3, 8 of the 11 bins within 0.698 rad of zero exceed the shuffle's 95th percentile and the above-chance fraction is 0.158. The remapping cells number 1,465 on day 3 and 1,500 on day 14, above the paper's 1,082 and 1,153.

## fig2h-rr-remapping-grows

> The above-chance fraction of RR remapping cells grew linearly with task experience over the seven switch days.

In [ ]:
y = np.array([AC[d][0] for d in sosa.SWITCH_DAYS]); x = np.arange(1, 8)
lr = st.linregress(x, y)
print("pooled per day:", dict(zip(sosa.SWITCH_DAYS, y.round(3))))
print(f"Figure 2h reading: slope {lr.slope:.4f} per switch, p = {lr.pvalue:.3g}")
yc = np.array([AC[d + 100][0] for d in sosa.SWITCH_DAYS]); lrc = st.linregress(x, yc)
print(f"code bins variant: slope {lrc.slope:.4f}, p = {lrc.pvalue:.3g}")
rng = np.random.default_rng(2002)
rows = []
for m in mice:
    for d in sosa.SWITCH_DAYS:
        rows.append(dict(mouse=m, switch=SWI[d], frac=above_chance([REM[(m, d)]], rng, nperm=1000)[0]))
PM = pd.DataFrame(rows)
b, p, n = mixed(PM, "frac", logit=True)
print(f"Extended Data Fig. 2h reading: per mouse-day logit mixed model slope {b:.3f}, p = {p:.3g}, n = {n} (dropped {len(PM) - n} at 0 or 1)")
b2, p2, n2 = mixed(PM, "frac")
print(f"   raw-fraction variant: slope {b2:.4f}, p = {p2:.3g}")
print(PM.pivot(index="mouse", columns="switch", values="frac").round(3))

**Result: reproduced.** The pooled above-chance fraction rises from 0.158 to 0.257 (slope 0.018 per switch, p = 0.0084), and the per-mouse logit mixed model has a positive slope (0.153, p = 0.0027).

## fig2eg-mean-rr-position-after-reward

> The mean reward-relative firing position of cells remapping within 50 cm relative to reward was significantly after the reward zone start, 0.441 rad on the first switch day (n = 453 cells) and 0.292 rad on the last (n = 645 cells).

In [ ]:
def rr_mean(d, excl=None, key=REM):
    rel0 = np.concatenate([key[(m, d)]["rel0"] for m in mice]); rel1 = np.concatenate([key[(m, d)]["rel1"] for m in mice])
    if excl is not None:
        ok = ~((np.abs(rel0) <= excl) & (np.abs(rel1) <= excl)); rel0, rel1 = rel0[ok], rel1[ok]
    kept = np.abs(sosa.wrap(rel1 - rel0)) <= RAD50 + 1e-9
    v = sosa.circmean(np.column_stack([rel0[kept], rel1[kept]]), axis=1)
    mu, lo, hi = sosa.circ_mtest_ci(v)
    return mu, lo, hi, kept.sum(), len(rel0)
for d, ev, en in [(3, 0.441, 453), (14, 0.292, 645)]:
    mu, lo, hi, n, ntot = rr_mean(d)
    print(f"day {d}: circular mean {mu:.3f} rad, 95% CI {lo:.3f} to {hi:.3f}, n = {n} of {ntot} (expected {ev}, n {en})")
for d in sosa.SWITCH_DAYS:
    mu, lo, hi, n, _ = rr_mean(d)
    print(f"   day {d:2d}: {mu:.3f} [{lo:.3f}, {hi:.3f}] n {n}")
mu, lo, hi, n, _ = rr_mean(3, key=REMc); print(f"code bins variant day 3: {mu:.3f} [{lo:.3f}, {hi:.3f}] n {n}")
mu, lo, hi, n, _ = rr_mean(14, key=REMc); print(f"code bins variant day 14: {mu:.3f} [{lo:.3f}, {hi:.3f}] n {n}")

**Result: reproduced.** The circular mean is 0.385 rad (95% CI 0.261 to 0.508) on day 3 and 0.321 rad (0.238 to 0.404) on day 14, both within tolerance. The cell counts (676 and 850) are above the paper's (453 and 645).

## fig2i-rr-remapping-spans-track

> After excluding cells with peaks within increasing distances on either side of both reward zone starts, above-chance RR remapping remained, spanning nearly the whole track by the last switch day.

In [ ]:
rng = np.random.default_rng(2003)
EX = {}
for d in sosa.SWITCH_DAYS:
    EX[d] = [above_chance([REM[(m, d)] for m in mice], rng, excl=dist / CM)[0] for dist in range(10, 200, 10)]
E = pd.DataFrame(EX, index=range(10, 200, 10)); E.index.name = "exclusion cm"
print(E.round(3).T)
for d in (3, 14):
    pos = E[d] > 0
    largest = max([x for x in E.index if pos[x]], default=0)
    allpos = int(pos.all())
    first_zero = next((x for x in E.index if not pos[x]), None)
    print(f"day {d}: positive at every distance 10-190 cm: {bool(allpos)}; positive up to (contiguous from 10 cm) "
          f"{(first_zero - 10) if first_zero else 190} cm; largest distance with a positive fraction {largest} cm")

**Result: reproduced.** On day 14 the above-chance fraction is positive at every exclusion distance from 10 to 190 cm (0.016 to 0.231). On day 3 it is positive up to 170 cm and again at 190 cm.

## ed2jl-rr-remapping-beyond-50cm

> Excluding cells within 50 cm of both reward zone starts, RR remapping still exceeded the shuffle, and the mean reward-relative position of the remaining RR remapping cells was 1.145 rad (n = 285) on the first switch and 1.245 rad (n = 303) on the last.

In [ ]:
rng = np.random.default_rng(2004)
for d, en, ev, ek in [(3, 888, 1.145, 285), (14, 795, 1.245, 303)]:
    fr, h, q95, n, bc, _, _ = above_chance([REM[(m, d)] for m in mice], rng, excl=RAD50)
    mu, lo, hi, k, ntot = rr_mean(d, excl=RAD50)
    print(f"day {d}: cells after exclusion {n} (exp {en}); above-chance fraction {fr:.3f}; "
          f"circular mean {mu:.3f} rad [{lo:.3f}, {hi:.3f}], n {k} (exp {ev}, n {ek})")

**Result: partially reproduced.** With the 50 cm exclusion the above-chance fraction is positive (0.044 on day 3, 0.077 on day 14), and the circular means (1.315 and 1.360 rad) are within tolerance with intervals that exclude zero. The cells remaining after exclusion (1,197 and 1,060) are outside the tolerance of 888 and 795.

## ed2m-more-fields-after-reward

> More RR cells fired after than before the reward zone start, after excluding cells within distances up to 50 cm on the first switch and up to 80 cm on the last.

In [ ]:
rows = []
for d in (3, 14):
    rel1 = np.concatenate([res[(m, d)]["circ"]["m"]["rel1"][CL[(m, d)]["rr"]] for m in mice])
    for dist in range(10, 200, 10):
        x = rel1[np.abs(rel1) > dist / CM]
        na = int((x > 0).sum()); n = len(x)
        z, p = proportions_ztest(na, n, value=0.5) if n > 0 else (np.nan, np.nan)
        rows.append(dict(day=d, excl_cm=dist, n=n, after=na, frac_after=na / n if n else np.nan, z=z, p=p,
                         sig=(p < 0.05 / 19) and (na / n > 0.5)))
Z = pd.DataFrame(rows)
print(Z.round(5).to_string(index=False))
ok3 = Z[(Z.day == 3) & (Z.excl_cm <= 50)].sig.all(); ok14 = Z[(Z.day == 14) & (Z.excl_cm <= 80)].sig.all()
print(f"day 3, after > 0.5 at p < 0.00263 for every distance 10-50 cm: {ok3}; day 14, 10-80 cm: {ok14}")

**Result: partially reproduced.** On day 3 the fraction after the zone start exceeds 0.5 at p < 0.00263 for every exclusion from 10 to 50 cm (and to 80 cm). On day 14 it does to 60 cm, but not at 70 cm (p = 0.0071) or 80 cm (p = 0.023).

## ed2n-far-remapping-no-increase

> After excluding cells within 50 cm of the reward zone start, the above-chance fraction of RR remapping cells did not increase significantly across days.

In [ ]:
rng = np.random.default_rng(2005)
y = np.array([above_chance([REM[(m, d)] for m in mice], rng, excl=RAD50)[0] for d in sosa.SWITCH_DAYS])
lr = st.linregress(np.arange(1, 8), y)
print("above-chance fraction beyond 50 cm by day:", dict(zip(sosa.SWITCH_DAYS, y.round(3))))
print(f"linear regression on switch 1-7: slope {lr.slope:.4f}, p = {lr.pvalue:.3g}")

**Result: reproduced.** Beyond 50 cm the above-chance fraction has no significant trend over switches (slope 0.0047, p = 0.175).

## ed2f-teleport-distance

> The distance run in the teleport zone did not predict the next lap's spatial firing error for ~93% of RR cells; 6.7% of RR cells had a significant Spearman correlation on trials before the switch and 7.5% after.

In [ ]:
out = {0: [], 1: []}
for k in CL:
    r = res[k]; c = r["circ"]["m"]; cen = c["centers"]
    rr = np.where(CL[k]["rr"])[0]
    td = r["beh"]["tele_dist"]
    for si, s in enumerate(r["sets"]):
        s = np.asarray(s)
        avg = c["peak0"] if si == 0 else c["peak1"]
        for ci in rr:
            tp = c["trial_peak"][s, ci]
            err = sosa.wrap(cen[np.clip(tp, 0, 44)] - cen[avg[ci]]) * CM
            err[tp < 0] = np.nan
            x = td[s[:-1]]; yv = err[1:]            # distance after trial t, error on trial t+1
            ok = np.isfinite(x) & np.isfinite(yv)
            if ok.sum() < 5:
                out[si].append(np.nan); continue
            out[si].append(st.spearmanr(x[ok], yv[ok]).pvalue)
for si, nm, ev in [(0, "before", 6.7), (1, "after", 7.5)]:
    p = np.array(out[si]); v = np.isfinite(p)
    print(f"{nm}: RR cells tested {v.sum()}, significant {100 * np.mean(p[v] < 0.05):.1f}% (expected {ev}%)")
print("RR cells pooled over mice and switch days:", sum(CL[k]["rr"].sum() for k in CL), "(expected 5979)")
print("teleport distance per trial, cm: median %.1f, range %.1f to %.1f" % tuple(
    np.nanpercentile(np.concatenate([res[k]["beh"]["tele_dist"] for k in CL]), [50, 0, 100])))

**Result: reproduced.** 6.1% of RR cells before and 6.8% after the switch have a significant Spearman correlation, both within tolerance. The pooled RR count (6,762) is above the paper's 5,979 +/- 598.

In [ ]:
# Sequences (Methods, Sequence detection and quantification): circular track coordinates,
# unsmoothed dF/F maps; positions are the peaks of the even pre-switch trials and of all post-switch
# trials; circular-circular correlation; 1,000 permutations of cell identities after the switch.
def seq_test(k, cat, rng, nperm=1000):
    c = res[k]["circ"]["m"]; cen = c["centers"]
    idx = np.where(CL[k][cat] & (c["peak_even"] >= 0) & (c["peak1"] >= 0))[0]
    if len(idx) < 5:
        return dict(n=len(idx), rho=np.nan, P=1.0)
    a = cen[c["peak_even"][idx]]; b = cen[c["peak1"][idx]]
    rho = sosa.corr_cc(a, b)
    sh = np.array([sosa.corr_cc(a, rng.permutation(b)) for _ in range(nperm)])
    P = (np.sum(np.abs(sh) >= abs(rho)) + 1) / (nperm + 1)
    return dict(n=len(idx), rho=rho, P=P)
rng = np.random.default_rng(4001)
SEQ = {cat: pd.DataFrame([dict(mouse=k[0], day=k[1], **seq_test(k, cat, rng)) for k in CL])
       for cat in ("rr", "tr", "nonrr")}

## fig4c-rr-sequences-preserved

> RR cells formed sequences whose firing order was preserved across the reward switch, above a cell-identity shuffle in 75 of 77 sessions.

In [ ]:
S = SEQ["rr"]
print(f"RR sequences above shuffle (P < 0.05): {int((S.P < 0.05).sum())} of {len(S)} (expected 75 +/- 7.5); "
      f"sessions with < 5 cells: {int((S.n < 5).sum())}")
print("rho median %.2f, range %.2f to %.2f; cells per session median %d" % (S.rho.median(), S.rho.min(), S.rho.max(), S.n.median()))
print("not significant:", S[S.P >= 0.05][["mouse", "day", "n", "rho", "P"]].round(3).values.tolist())

**Result: reproduced.** RR sequences exceed the identity shuffle in 75 of 77 sessions (median rho 0.73).

## fig4f-tr-sequences-preserved

> TR cells likewise formed sequences preserved across the switch, above shuffle in 76 of 77 sessions.

In [ ]:
S = SEQ["tr"]
print(f"TR sequences above shuffle: {int((S.P < 0.05).sum())} of {len(S)} (expected 76 +/- 7.6); < 5 cells: {int((S.n < 5).sum())}")
print("rho median %.2f; not significant:" % S.rho.median(), S[S.P >= 0.05][["mouse", "day", "n", "rho", "P"]].round(3).values.tolist())

**Result: reproduced.** TR sequences exceed the shuffle in 76 of 77 sessions (median rho 0.86).

## fig4i-nonrr-sequences-not-preserved

> Sequential firing was not preserved for most non-RR remapping cells; 61 of 77 sessions did not exceed the shuffle.

In [ ]:
S = SEQ["nonrr"]
print(f"non-RR sequences NOT above shuffle: {int((S.P >= 0.05).sum())} of {len(S)} (expected 61 +/- 6.1); < 5 cells: {int((S.n < 5).sum())}")
print("rho median %.2f, range %.2f to %.2f" % (S.rho.median(), S.rho.min(), S.rho.max()))
print("significant sessions by day:", S[S.P < 0.05].groupby("day").size().to_dict())

**Result: reproduced.** Non-RR remapping sequences do not exceed the shuffle in 62 of 77 sessions (expected 61 +/- 6.1; median rho -0.04).

In [ ]:
# Per mouse and day densities of RR post-switch relative peaks and other per-session measures
def lick_circvars(r, si):
    """Circular variance of licking positions relative to the zone start of trial set si:
    (a) all licks of the set pooled, (b) the code's circular mean over trials of each trial's
    circular variance."""
    zs = r["zone_starts"][si]
    s = r["sets"][si]
    lp = [sosa.wrap(sosa.cm_to_rad(r["beh"]["lick_pos"][t]) - sosa.cm_to_rad(zs)) for t in s
          if not r["beh"]["sensor_err"][t]]
    allp = np.concatenate(lp) if lp else np.array([])
    per = np.array([sosa.circvar(x) for x in lp if len(x) > 0])
    return sosa.circvar(allp), (sosa.circmean(per) if len(per) else np.nan)
DEN = []
for k in CL:
    r = res[k]; c = r["circ"]["m"]; C = CL[k]
    rr = C["rr"]; npc = C["pc"].sum()
    rel1 = c["rel1"][rr]; rel0 = c["rel0"][rr]
    va, vb = lick_circvars(r, 1); pa, pb = lick_circvars(r, 0)
    DEN.append(dict(mouse=k[0], day=k[1], switch=SWI[k[1]], npc=npc, n_rr=rr.sum(),
                    near=np.sum(np.abs(rel1) <= 50 / CM + 1e-9) / npc, far=np.sum(np.abs(rel1) > 50 / CM + 1e-9) / npc,
                    rr_var_post=sosa.circvar(rel1) if rr.sum() else np.nan,
                    rr_var_pre=sosa.circvar(rel0) if rr.sum() else np.nan,
                    rr_mean_post=sosa.circmean(rel1) if rr.sum() else np.nan,
                    lick_var_post_a=va, lick_var_post_b=vb, lick_var_pre_a=pa, lick_var_pre_b=pb))
DEN = pd.DataFrame(DEN)

## fig4j-rr-overrepresent-reward

> RR sequences spanned the whole track but over-represented a region from ~50 cm before to ~70 cm after the reward zone start, with the highest density following the start of the zone.

In [ ]:
# relative bins: post-switch peak bin minus the bin where the zone starts, so that bin 0 spans
# 0-10 cm after the zone start (peaks on the 2pi/45 grid fall on these bins exactly)
rel_cm_lo = 10 * np.arange(-22, 23)          # lower edge of each relative bin, cm
dens = {}
for k in CL:
    c = res[k]["circ"]["m"]; zb = int(res[k]["zone_starts"][1] // 10)
    rb = (c["peak1"][CL[k]["rr"]] - zb + 22) % 45     # 0..44 for relative bins -22..22
    dens[k] = np.bincount(rb, minlength=45) / CL[k]["pc"].sum()
def region(dmean, uni):
    above = dmean > uni
    j = int(np.argmax(dmean))
    lo = j
    while lo - 1 >= 0 and above[lo - 1]: lo -= 1
    hi = j
    while hi + 1 < 45 and above[hi + 1]: hi += 1
    return rel_cm_lo[lo], rel_cm_lo[hi] + 10, rel_cm_lo[j], above.sum()
def report(days, label):
    dm = np.mean([np.mean([dens[(m, d)] for m in mice], 0) for d in days], 0)
    uni = np.mean([np.mean([CL[(m, d)]["rr"].sum() / CL[(m, d)]["pc"].sum() for m in mice]) for d in days]) / 45
    lo, hi, pk, nab = region(dm, uni)
    ok = abs(lo - (-50)) <= 10 and abs(hi - 70) <= 10 and pk >= 0
    print(f"{label}: contiguous region above uniform around the densest bin, from {lo:.0f} to {hi:.0f} cm "
          f"(outer bin edges; expected -50 and 70 +/- 10); densest bin {pk:.0f} to {pk + 10:.0f} cm; bins above uniform {nab}; "
          f"uniform {uni:.4f}; max density {dm.max():.4f}; matches: {ok}")
    return dm, uni
dm, uni = report(sosa.SWITCH_DAYS, "(a) mean over the 7 switch days")
for d in sosa.SWITCH_DAYS:
    report([d], f"(b) day {d:2d}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(rel_cm_lo + 5, dm, "k.-"); ax.axhline(uni, color="r"); ax.axvline(0, color="0.5")
ax.set_xlabel("RR post-switch peak relative to zone start (cm)"); ax.set_ylabel("fraction of place cells")
plt.show()
print("plotted density range %.4f to %.4f, all bins shown" % (dm.min(), dm.max()))

**Result: partially reproduced.** The densest bin follows the zone start (10 to 20 cm after it), and the region above the uniform level starts 50 cm before the zone start, but it ends 90 cm after it, outside the 70 +/- 10 cm expected. No single day matches both limits.

## fig4m-rr-density-near-reward-grows

> The density of RR cells within 50 cm of the reward zone start increased with task experience, while the fraction of place cells more than 50 cm from it stayed stable over days.

In [ ]:
for lg in (False, True):
    bn, pn, nn = mixed(DEN, "near", logit=lg); bf, pf, nf = mixed(DEN, "far", logit=lg)
    print(f"{'(b) logit' if lg else '(a) raw'}: near slope {bn:.4f} p = {pn:.3g} (n {nn}); far slope {bf:.4f} p = {pf:.3g} (n {nf})")
print(DEN.groupby("day")[["near", "far"]].mean().round(4))

**Result: reproduced.** The fraction of place cells that are RR within 50 cm of the zone start increases (raw slope 0.0063, p = 3.6e-5; logit p = 0.00019), and the fraction beyond 50 cm does not change significantly (p = 0.083 raw, 0.062 logit).

## fig4no-licking-speed-increase

> Over switch days mice increased their post-switch anticipatory lick rate and their running speed.

In [ ]:
rows = []
for k in CL:
    r = res[k]; b = r["beh"]; s1 = r["sets"][1]; zs = r["zone_starts"][1]
    good = [t for t in s1 if not b["sensor_err"][t]]
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.nanmean(b["lick_cnt"][good] / np.where(b["occ_t"][good] > 0, b["occ_t"][good], np.nan), 0)
    c10 = 5 + 10 * np.arange(45); c2 = 1 + 2 * np.arange(225)
    spd = np.nanmean(b["speed2"][s1], 0)
    rows.append(dict(mouse=k[0], switch=SWI[k[1]], lick_near=np.nanmean(rate[np.abs(c10 - zs) <= 50]),
                     speed_near=np.nanmean(spd[np.abs(c2 - zs) <= 50]), speed_far=np.nanmean(spd[np.abs(c2 - zs) > 50])))
LS = pd.DataFrame(rows)
for y in ("lick_near", "speed_near", "speed_far"):
    b_, p_, n_ = mixed(LS, y)
    print(f"{y:11s}: slope {b_:.3f} per switch, p = {p_:.3g}, n = {n_}")
print(LS.groupby("switch")[["lick_near", "speed_near", "speed_far"]].mean().round(2))

**Result: reproduced.** The post-switch lick rate near the zone rises over switches (slope 0.148 Hz per switch, p = 0.00011), and so does speed, within 50 cm (p = 0.012) and beyond (p = 3.3e-6).

## fig4p-lick-precision-sequence-precision

> On a given day, the circular variance of the animal's licking positions correlated with the circular variance of the RR sequence, both before and after the switch.

In [ ]:
for rd in ("a", "b"):
    for ph, rv, lv in (("before", "rr_var_pre", f"lick_var_pre_{rd}"), ("after", "rr_var_post", f"lick_var_post_{rd}")):
        d = DEN[[rv, lv]].dropna()
        lr = st.linregress(d[lv], d[rv])
        print(f"reading ({rd}) {ph:6s}: slope {lr.slope:.3f}, r = {lr.rvalue:.3f}, p = {lr.pvalue:.3g}, n = {len(d)} sessions")
print("ranges: RR variance %.2f..%.2f; lick variance (a) %.2f..%.2f, (b) %.2f..%.2f" % (
    DEN.rr_var_post.min(), DEN.rr_var_post.max(), DEN.lick_var_post_a.min(), DEN.lick_var_post_a.max(),
    DEN.lick_var_post_b.min(), DEN.lick_var_post_b.max()))

**Result: reproduced under reading (a).** With all licks of the set pooled, the RR sequence variance rises with the lick variance before (p = 0.0032) and after the switch (p = 0.042). Under the code's per-trial reading (b), the after-switch slope is not significant (p = 0.067).

## ed4hij-sequence-variance-decreases

> The variance of the RR sequence and of post-switch licking positions decreased across switch days, while the mean reward-relative position of the sequence did not change.

In [ ]:
for nm, D_ in (("mouse-days without RR cells dropped", DEN), ("code: filled with 0", DEN.fillna({"rr_var_post": 0, "rr_mean_post": 0}))):
    b1, p1, n1 = mixed(D_, "rr_var_post"); b3, p3, n3 = mixed(D_, "rr_mean_post")
    print(f"{nm}: RR variance slope {b1:.4f} p = {p1:.3g} (n {n1}); RR mean position slope {b3:.4f} p = {p3:.3g}")
for rd in ("a", "b"):
    b2, p2, n2 = mixed(DEN, f"lick_var_post_{rd}")
    print(f"post-switch lick variance, reading ({rd}): slope {b2:.4f} p = {p2:.3g} (n {n2})")
print(DEN.groupby("day")[["rr_var_post", "rr_mean_post", "lick_var_post_a", "lick_var_post_b"]].mean().round(3))

**Result: partially reproduced.** Post-switch lick variance decreases (p = 0.0032 and 0.0018 under the two readings), and the mean RR position does not change (p = 0.50), but the RR sequence variance falls without significance (slope -0.012, p = 0.069).

## fig4qr-tr-track-ends

> TR sequences tended to over-represent the ends of the track, and TR density far from the track ends decreased across days.

In [ ]:
rows = []; dens_tr = []
for k in CL:
    c = res[k]["circ"]["m"]; tr = CL[k]["tr"] & (c["peak1"] >= 0); npc = CL[k]["pc"].sum()
    pos_cm = (c["centers"][c["peak1"][tr]] + np.pi) * CM
    dens_tr.append(np.bincount(c["peak1"][tr], minlength=45) / npc)
    far = np.sum((pos_cm > 50) & (pos_cm < 400)) / npc
    rows.append(dict(mouse=k[0], switch=SWI[k[1]], far=far, frac_tr=tr.sum() / npc))
TRD = pd.DataFrame(rows)
dmean = np.mean(dens_tr, 0); uni = TRD.frac_tr.mean() / 45
cpos = (res[next(iter(CL))]["circ"]["m"]["centers"] + np.pi) * CM
ends = (cpos <= 50) | (cpos >= 400)
print(f"mean TR density in bins within 50 cm of a track end {dmean[ends].mean():.4f} vs uniform {uni:.4f}; elsewhere {dmean[~ends].mean():.4f}")
for lg in (False, True):
    b_, p_, n_ = mixed(TRD, "far", logit=lg)
    print(f"{'logit' if lg else 'raw'}: TR fraction > 50 cm from both ends, slope {b_:.4f}, p = {p_:.3g}, n = {n_}")
print(TRD.groupby("switch").far.mean().round(4).to_dict())

**Result: reproduced.** TR density within 50 cm of the track ends (0.0103 per bin) exceeds the uniform level (0.0047), and the fraction of place cells that are TR cells far from the ends decreases (raw slope -0.0072, p = 0.0053; logit p = 0.0098).

## ed4l-disappearing-decrease

> Disappearing, appearing and non-RR remapping cells encoded a more uniform representation; the fraction of disappearing cells decreased with experience both near and far from reward, while non-RR remapping and appearing fractions did not change significantly.

In [ ]:
rows = []
for k in CL:
    r = res[k]; c = r["circ"]["m"]; C = CL[k]; npc = C["pc"].sum()
    row = dict(mouse=k[0], switch=SWI[k[1]])
    # disappearing cells: their pre-switch peak relative to the pre-switch zone; others: post-switch
    for cat, rel in (("dis", c["rel0"]), ("app", c["rel1"]), ("nonrr", c["rel1"])):
        x = rel[C[cat]]
        row[cat + "_near"] = np.sum(np.abs(x) <= 50 / CM + 1e-9) / npc
        row[cat + "_far"] = np.sum(np.abs(x) > 50 / CM + 1e-9) / npc
    rows.append(row)
CAT = pd.DataFrame(rows)
for col in [c for c in CAT.columns if c.endswith(("_near", "_far"))]:
    b_, p_, n_ = mixed(CAT, col)
    print(f"{col:11s}: slope {b_:+.4f}, p = {p_:.3g}")

**Result: partially reproduced.** Disappearing cells near reward decrease (slope -0.0015, p = 0.040), but far from reward the decrease is not significant (p = 0.21). Non-RR remapping and appearing fractions show no significant change in either window (p from 0.063 to 0.26).

## ed2cd-switch-remapping-intermediate

> A reward switch within a constant environment induced more remapping than a stay day with a fixed reward and environment, and less than a reward switch combined with a novel environment.

In [ ]:
def remap2d(k, band="legend"):
    r = res[k]; pc0, pc1 = r["sig"]; both = pc0 & pc1 & ~r["is_int"]
    p0, p1 = r["peak_lin"][0][both], r["peak_lin"][1][both]
    z0, z1 = r["zone_starts"]
    near = (np.abs(p0 - z0) <= 50) & (np.abs(p1 - z1) <= 50)
    if band == "legend":
        diag = (np.abs(p1 - p0) <= 50) & ~near
    else:   # the code's np.tri band, offsets of -4 to +5 bins (post minus pre)
        diag = ((p1 - p0) >= -40) & ((p1 - p0) <= 50) & ~near
    n = both.sum()
    return dict(diag=diag.sum() / n, near=near.sum() / n, other=(~diag & ~near).sum() / n, n=n)
def kind(d):
    return "stay" if d in sosa.STAY_DAYS else ("across" if d == 8 else "within")
for band in ("legend", "code"):
    R = pd.DataFrame([dict(mouse=m, day=d, kind=kind(d), **remap2d((m, d), band)) for (m, d) in res])
    M = R.groupby(["mouse", "kind"])[["diag", "other"]].mean().unstack()
    print(f"diagonal band: {band}; mouse means (fractions of place cells significant in both sets):")
    print(M.round(3))
    for cat in ("diag", "other"):
        L = np.log(M[cat] / (1 - M[cat]))
        comps = [("stay", "within"), ("stay", "across"), ("within", "across")]
        ts = [st.ttest_rel(L[a], L[b]) for a, b in comps]
        ph = holm([t.pvalue for t in ts])
        for (a, b), t, p in zip(comps, ts, ph):
            print(f"   {cat:5s} {a:6s} vs {b:6s}: means {M[cat][a].mean():.3f} vs {M[cat][b].mean():.3f}, t = {t.statistic:.1f}, Holm p = {p:.2g}")
    if band == "legend":
        RL = R

**Result: reproduced.** All other remapping is ordered stay (0.268) < switch within (0.408) < across environment (0.587), and the diagonal fraction is ordered stay (0.535) > switch within (0.491) > across (0.300), each Holm p < 0.05 (largest 0.04). The code's diagonal band gives the same order (largest p 0.048).

In [ ]:
# Cross-day matching of ROIs (Methods, Calcium data processing): intersection over union of ROI
# pixels, with the threshold chosen per pair so that every best match exceeds every second-best
# match. Before the IoU, day B's ROIs are shifted by the rigid offset between the two days'
# mean images (phase correlation), an assumption: the Methods say the same field of view was
# acquired each day by aligning to a reference image, and the matching code is in TwoPUtils,
# which the paper's repository does not hold.
import scipy.sparse as sps
def phase_shift(a, b):
    A = np.fft.fft2((a - a.mean()) / a.std()); Bf = np.fft.fft2((b - b.mean()) / b.std())
    R = A * np.conj(Bf); R /= np.abs(R) + 1e-12
    c = np.abs(np.fft.ifft2(R))
    dy, dx = np.unravel_index(np.argmax(c), c.shape)
    H, W = a.shape
    if dy > H // 2: dy -= H
    if dx > W // 2: dx -= W
    return int(dy), int(dx)
def match_pair(ka, kb):
    ra, rb = res[ka], res[kb]
    H, W = ra["mean_imgs"][0].shape
    npl = len(ra["mean_imgs"])
    shifts = [phase_shift(ra["mean_imgs"][p], rb["mean_imgs"][p]) for p in range(npl)]
    def mat(r, shift=None):
        rows, cols = [], []
        for i, px in enumerate(r["pix"]):
            z = px // (H * W); y = (px % (H * W)) // W; x = px % W     # y image row, x column
            if shift is not None:
                dy = np.array([shifts[int(zz)][0] for zz in z]); dx = np.array([shifts[int(zz)][1] for zz in z])
                y = y + dy; x = x + dx
                ok = (y >= 0) & (y < H) & (x >= 0) & (x < W); z, y, x = z[ok], y[ok], x[ok]
            rows.append(np.full(len(x), i)); cols.append(z * H * W + y * W + x)
        rows = np.concatenate(rows); cols = np.concatenate(cols)
        return sps.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(r["pix"]), npl * H * W))
    A = mat(ra); B = mat(rb, shift=True)
    inter = (A @ B.T).toarray()
    sa = np.asarray(A.sum(1)).ravel(); sb = np.asarray(B.sum(1)).ravel()
    iou = inter / (sa[:, None] + sb[None, :] - inter + 1e-12)
    srt_r = np.sort(iou, 1); srt_c = np.sort(iou, 0)
    thr = max(srt_r[:, -2].max(), srt_c[-2, :].max())
    best_b = iou.argmax(1); best_a = iou.argmax(0)
    ia = np.arange(iou.shape[0])
    mutual = best_a[best_b] == ia
    keep = mutual & (iou[ia, best_b] > thr)
    return ia[keep], best_b[keep], thr, shifts
PAIRS = list(zip(sosa.SWITCH_DAYS[:-1], sosa.SWITCH_DAYS[1:]))
MATCH = {}
for m in mice:
    for i, (da, db) in enumerate(PAIRS):
        MATCH[(m, i)] = match_pair((m, da), (m, db))
mt = pd.DataFrame([dict(mouse=m, pair=f"{PAIRS[i][0]}-{PAIRS[i][1]}", matched=len(v[0]),
                        cells_a=res[(m, PAIRS[i][0])]["n_cells"], thr=v[2], shift=v[3][0]) for (m, i), v in MATCH.items()])
mt["frac"] = mt.matched / mt.cells_a
print("matched fraction of day-A cells: median %.2f, range %.2f to %.2f; IoU threshold median %.2f, range %.2f to %.2f" % (
    mt.frac.median(), mt.frac.min(), mt.frac.max(), mt.thr.median(), mt.thr.min(), mt.thr.max()))
print("rigid shifts (dy, dx) between days, largest |shift|: %d pixels" % max(max(abs(s[0]), abs(s[1])) for s in mt["shift"]))
print(mt.pivot(index="mouse", columns="pair", values="frac").round(2))

In [ ]:
def transitions(src, dst, min_n=5):
    rows = []
    for (m, i), (ia, ib, thr, sh) in MATCH.items():
        da, db = PAIRS[i]
        A, B = CL[(m, da)], CL[(m, db)]
        a_src = A[src][ia] if src != "nontr" else ((A["app"] | A["dis"] | A["nonrr"]) & ~A["rr"])[ia]
        b_pc = B["pc"][ib]
        den = (a_src & b_pc).sum()
        num = (a_src & b_pc & B[dst][ib]).sum()
        rows.append(dict(mouse=m, pair=i, num=num, den=den, frac=num / den if den >= min_n and den > 0 else np.nan))
    return pd.DataFrame(rows)
def trans_report(src, dst, min_n=5, label=""):
    T = transitions(src, dst, min_n)
    b_, p_, n_ = mixed(T, "frac", x="pair")
    print(f"{label} {src} -> {dst}: slope {b_:+.4f} per pair, p = {p_:.3g}, n = {n_} mouse-pairs; "
          f"pooled {T.num.sum()}/{T.den.sum()}; mean by pair {T.groupby('pair').frac.mean().round(3).tolist()}")
    return b_, p_

## fig5b-rr-remain-rr-increases

> Across experience, RR cells were increasingly likely to remain RR on the next switch day.

In [ ]:
trans_report("rr", "rr")

**Result: reproduced.** The fraction of RR cells staying RR rises over day pairs (slope 0.021 per pair, p = 0.0019).

## fig5d-nontr-to-rr-increases

> An increasing fraction of non-TR place cells (appearing, disappearing and non-RR remapping) converted into RR cells on the next switch day over experience.

In [ ]:
trans_report("nontr", "rr")

**Result: not reproduced.** The fraction of non-TR place cells becoming RR has a positive but non-significant slope (0.0070, p = 0.137).

## fig5f-nonpc-to-rr-increases

> The fraction of non-place cells converting to RR cells on the next switch day showed a modest increase over experience.

In [ ]:
trans_report("nonpc", "rr")

**Result: not reproduced.** The fraction of non-place cells becoming RR has a positive but non-significant slope (0.0058, p = 0.20).

## fig5h-tr-to-rr-constant

> TR cells were sometimes recruited into the RR population, at a rate that did not change over days.

In [ ]:
trans_report("tr", "rr")

**Result: reproduced.** TR cells become RR at a rate with no significant trend (slope 0.0025, p = 0.57; pooled 542 of 4,501).

## fig5i-rr-to-tr-decreases

> The fraction of RR cells converting to TR cells decreased slightly over experience.

In [ ]:
trans_report("rr", "tr", min_n=1)

**Result: reproduced.** The fraction of RR cells becoming TR decreases (slope -0.016 per pair, p = 0.027).

## fig5jkl-tr-no-recruitment

> The TR population did not recruit more cells from non-TR place cells or non-place cells over days, and TR cells were not more likely to remain TR across experience.

In [ ]:
for s in ("nontr", "nonpc", "tr"):
    trans_report(s, "tr")

**Result: reproduced.** None of the three TR recruitment slopes is positive at p < 0.05 (non-TR to TR -0.0057, p = 0.35; non-place to TR -0.0039, p = 0.52; TR to TR 0.0021, p = 0.79).

## ed6a-rr-near-reward-stable

> RR cells that first fired closer to reward were more likely to keep stable RR tuning across days, and TR cells closer to reward were more likely to remap relative to the track and become RR.

In [ ]:
rr_d, rr_c, tr_d, tr_c = [], [], [], []
for (m, i), (ia, ib, thr, sh) in MATCH.items():
    da, db = PAIRS[i]
    ca, cb = res[(m, da)]["circ"]["m"], res[(m, db)]["circ"]["m"]
    cen = ca["centers"]
    A = CL[(m, da)]
    sel = A["rr"][ia]
    rr_d.append(np.abs(ca["rel0"][ia][sel]) * CM)
    rr_c.append(np.abs(sosa.wrap(cb["rel0"][ib][sel] - ca["rel0"][ia][sel])) * CM)
    sel = A["tr"][ia]
    tr_d.append(np.abs(ca["rel0"][ia][sel]) * CM)
    tr_c.append(np.abs(sosa.wrap(cen[cb["peak0"][ib][sel]] - cen[ca["peak0"][ia][sel]])) * CM)
rr_d, rr_c, tr_d, tr_c = map(np.concatenate, (rr_d, rr_c, tr_d, tr_c))
print(f"followed cells: RR {len(rr_d)} (expected 2294 +/- 229), TR {len(tr_d)} (expected 3408 +/- 341)")
for nm, d, c in (("RR reward-relative change", rr_d, rr_c), ("TR track change", tr_d, tr_c)):
    near, far = c[d <= 50], c[d > 50]
    z, p = st.ranksums(near, far)
    lr = st.linregress(d, c)
    print(f"{nm}: (a) near {np.mean(near):.1f} cm (n {len(near)}) vs far {np.mean(far):.1f} cm (n {len(far)}), Z = {z:.2f}, p = {p:.3g}; "
          f"(b) slope {lr.slope:.3f} cm/cm, p = {lr.pvalue:.3g}")
    bins = np.arange(0, 230, 10); idx = np.digitize(d, bins) - 1
    print("   binned mean change by day-A distance (10 cm bins from 0):", [round(float(np.mean(c[idx == b])), 1) if np.any(idx == b) else None for b in range(len(bins) - 1)])

**Result: reproduced.** Both readings agree. RR cells near reward change their reward-relative position less (52.1 against 86.0 cm, p = 1.9e-71; slope 0.29 cm per cm, p = 4.3e-53), and TR cells near reward change their track position more (98.0 against 63.4 cm, p = 2.6e-49; slope -0.21, p = 1.0e-43). The followed cells (RR 3,407, TR 5,217) are more than the paper's 2,294 and 3,408.

## ed6c-rr-shift-toward-reward

> RR cells that stayed RR across switch-day pairs shifted their peaks slightly toward the reward zone start over days, their initial position being weakly anticorrelated with the change.

In [ ]:
x_, y_ = [], []
for (m, i), (ia, ib, thr, sh) in MATCH.items():
    da, db = PAIRS[i]
    ca, cb = res[(m, da)]["circ"]["m"], res[(m, db)]["circ"]["m"]
    sel = CL[(m, da)]["rr"][ia] & CL[(m, db)]["rr"][ib]
    x0 = ca["rel0"][ia][sel]; dx = sosa.wrap(cb["rel0"][ib][sel] - x0)
    k = np.abs(x0) * CM <= 80
    x_.append(x0[k] * CM); y_.append(dx[k] * CM)
x_, y_ = np.concatenate(x_), np.concatenate(y_)
lr = st.linregress(x_, y_)
print(f"RR cells RR on both days within 80 cm: n = {len(x_)} (expected 551 +/- 55); slope {lr.slope:.3f}, r = {lr.rvalue:.3f}, p = {lr.pvalue:.3g}")

**Result: not reproduced.** The 577 cells (within tolerance of 551) have a negative slope that is not significant (-0.095, r = -0.073, p = 0.080).

## ed6-across-day-drift

> Across-day sequence correlations of RR and TR cells were generally lower than within-day sequence correlations, consistent with representational drift.

In [ ]:
def across(cat, keep_b):
    out = []
    for (m, i), (ia, ib, thr, sh) in MATCH.items():
        da, db = PAIRS[i]
        ca, cb = res[(m, da)]["circ"]["m"], res[(m, db)]["circ"]["m"]
        sel = CL[(m, da)][cat][ia]
        if keep_b:
            sel = sel & CL[(m, db)][cat][ib]
        sel = sel & (ca["peak1"][ia] >= 0) & (cb["peak0"][ib] >= 0)
        if sel.sum() >= 5:
            out.append(sosa.corr_cc(ca["centers"][ca["peak1"][ia][sel]], cb["centers"][cb["peak0"][ib][sel]]))
    return np.array(out)
for cat in ("rr", "tr"):
    within = SEQ[cat].rho.median()
    for rd, kb in (("(a) followed regardless of day-B class", False), ("(b) same class on day B", True)):
        a = across(cat, kb)
        print(f"{cat.upper()} {rd}: across-day median rho {np.median(a):.3f} over {len(a)} mouse-pairs vs within-day median {within:.3f}; lower: {np.median(a) < within}")

**Result: reproduced.** Across-day correlations are lower than within-day ones under both cell readings: RR 0.266 and 0.568 against 0.726, TR 0.231 and 0.543 against 0.857.

## ed7-no-anatomical-bias

> RR and TR cells showed no reliable bias in their medio-lateral, antero-posterior or superficial-deep distribution within CA1.

In [ ]:
rows = []
for m in mice:
    planes = sorted(set(np.concatenate([res[(m, d)]["plane"] for d in sosa.SWITCH_DAYS])))
    for p in planes:
        xs = {"rr": [], "tr": []}; ys = {"rr": [], "tr": []}
        for d in sosa.SWITCH_DAYS:
            r = res[(m, d)]
            for cat in ("rr", "tr"):
                sel = CL[(m, d)][cat] & (r["plane"] == p)
                xs[cat].append(r["cx"][sel]); ys[cat].append(r["cy"][sel])
        xr, xt = np.concatenate(xs["rr"]), np.concatenate(xs["tr"]); yr, yt = np.concatenate(ys["rr"]), np.concatenate(ys["tr"])
        zx, px = st.ranksums(xr, xt); zy, py = st.ranksums(yr, yt)
        rows.append(dict(mouse=m, plane=p, n_rr=len(xr), n_tr=len(xt), z_x=zx, p_x=px, z_y=zy, p_y=py))
AN = pd.DataFrame(rows)
print(AN.round(4).to_string(index=False))
nx, ny = int((AN.p_x < 0.0038).sum()), int((AN.p_y < 0.0038).sum())
print(f"mice or planes: {len(AN)}; significant at p < 0.0038: image x axis {nx}, image y axis {ny}")
print("with x as ML and y as AP: ML %d (limit 3), AP %d (limit 2); with x as AP and y as ML: ML %d, AP %d" % (nx, ny, ny, nx))
print("uncorrected p < 0.05: x %d, y %d" % ((AN.p_x < 0.05).sum(), (AN.p_y < 0.05).sum()))

**Result: partially reproduced.** In the two image axes, 2 and 0 of 13 mice or planes differ between RR and TR at p < 0.0038, within the limits under either assignment of the axes to ML and AP. The depth part cannot be tested, as the files hold no z-stack or depth estimate.

In [ ]:
# Time warping (Methods, Analysis of rewarded versus omission trials; Time warp modeling; code,
# Fig6_Ext8_Rewarded_vs_Omission.md:140-420): per trial set with the zone at A or B and at least
# three omission trials, a PiecewiseWarping model with three knots (warp_reg_scale 1e-6,
# smoothness_reg_scale 20, 50 iterations, 200 warp iterations) is fit to the trials' speed in 10 cm
# bins and applied to each RR cell's binned deconvolved activity divided by its mean.
from affinewarp import PiecewiseWarping
from sklearn.impute import KNNImputer
def omission_trials(r, s, zs):
    """Unrewarded trials in which the mouse licked inside the reward zone: an omission, not a
    miss (assumption: the files flag rewards but not omissions)."""
    b = r["beh"]
    zb = slice(int(zs // 10), int(zs // 10) + 5)
    licked = b["lick_cnt"][s][:, zb].sum(1) > 0
    return (~b["rewarded"][s]) & licked & ~b["sensor_err"][s]
def warp_set(k, si, rng_seed):
    r = res[k]; s = np.asarray(r["sets"][si]); zs = r["zone_starts"][si]
    if r["zones"][si] not in ("A", "B"):
        return None
    om = omission_trials(r, s, zs); rw = r["beh"]["rewarded"][s]
    if om.sum() < 3:
        return None
    spd = np.nanmean(r["beh"]["speed2"][s].reshape(len(s), 45, 5), 2)
    if np.any(np.isnan(spd)):
        spd = KNNImputer(n_neighbors=3).fit_transform(spd)
    c = r["circ"]["m"]
    pc_idx = r["pc_idx"]
    rr = CL[k]["rr"][pc_idx]
    relp = (c["rel0"] if si == 0 else c["rel1"])[pc_idx]
    pk_lin = r["peak_lin"][si][pc_idx]
    sel_b = rr & (relp > 0)                          # code: positive circular offset
    sel_a = rr & (pk_lin >= zs)                      # Methods: track peak between zone start and track end
    X = r["tm_dec_pc"][s].astype(float)              # trials x bins x place cells
    mu = np.nanmean(X.reshape(-1, X.shape[2]), 0)
    X = np.nan_to_num(X / mu)
    np.random.seed(rng_seed)
    sosa.seed_numba(rng_seed)
    model = PiecewiseWarping(n_knots=3, warp_reg_scale=1e-6, smoothness_reg_scale=20.0)
    model.fit(spd[:, :, None], iterations=50, warp_iterations=200, verbose=False)
    Xw = model.transform(X)
    sw = model.transform(spd[:, :, None])[:, :, 0]
    def mse(a):
        return np.mean((a - a.mean(0)) ** 2)
    out = dict(mouse=k[0], day=k[1], set=si, n_om=int(om.sum()), mse_raw=mse(spd), mse_warp=mse(sw))
    for nm, sel in (("a", sel_a), ("b", sel_b)):
        rmean = Xw[rw][:, :, sel].mean(0); omean = Xw[om][:, :, sel].mean(0)
        with np.errstate(invalid="ignore", divide="ignore"):
            out["ro_" + nm] = (rmean - omean).sum(0) / (rmean + omean).sum(0)
    return out
# affinewarp's warp search runs in numba threads with their own random generators; with one
# thread and the generator seeded, the fits repeat exactly
import numba
n_threads = numba.get_num_threads()
numba.set_num_threads(1)
WARP = []
for i, k in enumerate(sorted(CL)):
    for si in (0, 1):
        o = warp_set(k, si, 6000 + 10 * i + si)
        if o is not None:
            WARP.append(o)
numba.set_num_threads(n_threads)
print("trial sets warped: before %d, after %d" % (sum(w["set"] == 0 for w in WARP), sum(w["set"] == 1 for w in WARP)))
def ro_per_mouse(si, rd):
    rows = []
    for m in mice:
        v = np.concatenate([w["ro_" + rd] for w in WARP if w["mouse"] == m and w["set"] == si] or [np.array([])])
        v = v[np.isfinite(v)]
        nd = sum(1 for w in WARP if w["mouse"] == m and w["set"] == si)
        p = st.wilcoxon(v).pvalue if len(v) >= 5 else np.nan
        rows.append(dict(mouse=m, days=nd, cells=len(v), median=np.median(v) if len(v) else np.nan, p=p,
                         sig_pos=bool(len(v) >= 5 and p < 0.0045 and np.median(v) > 0)))
    return pd.DataFrame(rows)

## fig6e-rr-prefer-reward

> RR cells with peaks after the reward zone start fired more after rewards than after omissions before the switch; the median reward versus omission index was above 0 in every mouse and significantly so in 9 of 11 mice.

In [ ]:
for rd, nm in (("a", "Methods: track peak after the zone start"), ("b", "code: positive circular offset")):
    T = ro_per_mouse(0, rd)
    print(f"reading ({rd}) {nm}: mice with median > 0 at p < 0.0045: {int(T.sig_pos.sum())} of {len(T)} (expected 9 +/- 1); "
          f"medians > 0: {int((T['median'] > 0).sum())}")
    print(T.round(5).to_string(index=False))

**Result: partially reproduced.** The median reward-versus-omission index is above 0 in 10 of 11 mice under both readings, but significant at p < 0.0045 in only 5 (reading a) or 7 (reading b), below the 9 +/- 1 expected. The sets warped per mouse match the paper's day counts exactly (41 sets).

## ed8k-rr-prefer-reward-after-switch

> After the switch too, RR cells fired more following rewards than omissions, with a positive median reward versus omission index significant in 9 of 11 mice.

In [ ]:
for rd in ("a", "b"):
    T = ro_per_mouse(1, rd)
    print(f"reading ({rd}): mice with median > 0 at p < 0.0045: {int(T.sig_pos.sum())} of {len(T)} (expected 9 +/- 1); medians > 0: {int((T['median'] > 0).sum())}")
    print(T.round(5).to_string(index=False))

**Result: reproduced.** After the switch, 8 of 11 mice have a positive median at p < 0.0045 under both readings (within 9 +/- 1), and all 11 medians are positive. The sets warped per mouse match the paper's day counts (48 sets).

## ed8df-time-warp-speed-mse

> Time warping reduced the mean squared error across trial-by-trial speed profiles, and the speed error was higher after the switch than before.

In [ ]:
WD = pd.DataFrame([{k: v for k, v in w.items() if not k.startswith("ro_")} for w in WARP])
b0 = WD[WD.set == 0]
print(f"before-switch sets: {len(b0)} (paper 61 sessions); raw MSE median {b0.mse_raw.median():.1f}, warped {b0.mse_warp.median():.1f}, "
      f"Wilcoxon p = {st.wilcoxon(b0.mse_raw, b0.mse_warp).pvalue:.3g}; warped lower in {int((b0.mse_warp < b0.mse_raw).sum())}")
both = b0.merge(WD[WD.set == 1], on=["mouse", "day"], suffixes=("_pre", "_post"))
print(f"sessions with both sets warped: {len(both)}; raw MSE after {both.mse_raw_post.median():.1f} vs before {both.mse_raw_pre.median():.1f}, "
      f"Wilcoxon p = {st.wilcoxon(both.mse_raw_post, both.mse_raw_pre).pvalue:.3g}; after higher in {int((both.mse_raw_post > both.mse_raw_pre).sum())}")
# the paper's raw MSE does not need the warp: the speed MSE of every session, before and after
rows = []
for k in CL:
    r = res[k]
    v = []
    for si in (0, 1):
        s = np.asarray(r["sets"][si])
        spd = np.nanmean(r["beh"]["speed2"][s].reshape(len(s), 45, 5), 2)
        v.append(np.nanmean((spd - np.nanmean(spd, 0)) ** 2))
    rows.append(dict(mouse=k[0], day=k[1], pre=v[0], post=v[1]))
AM = pd.DataFrame(rows)
print(f"variant, all 77 sessions: raw speed MSE after {AM.post.median():.1f} vs before {AM.pre.median():.1f}, Wilcoxon p = {st.wilcoxon(AM.post, AM.pre).pvalue:.3g}")

**Result: reproduced.** Warping lowers the speed MSE in all 41 before-switch sets (median 141.5 to 39.3, p = 9.1e-13), and raw MSE is higher after than before the switch (180.2 against 132.3 over the 18 sessions with both sets warped, p = 0.00053; 174.0 against 139.6 over all 77, p = 0.00041).

In [ ]:
# Individual place fields (Methods, Place field definition and inclusion criteria; Formation lap
# identification and backwards shifting), from the per-trial binned deconvolved events of the
# place cells (10 cm bins, speed >= 2 cm/s). The Methods test activity and centres of mass on the
# raw, unbinned deconvolved activity; here they use the 10 cm bins (a deviation).
import scipy.ndimage as ndi
def fields_of(mat):
    """mat: trials x bins (30 trials) for one cell. Fields: runs of >= 2 bins (20 cm) where the
    trial-averaged activity smoothed by a 10 cm Gaussian exceeds 20% of its maximum, kept when the
    cell is active (binned activity > mean + 1 s.d. over the 30 trials) inside the field on >= 8
    trials. Returns a list of (start, stop) bin ranges."""
    mat = np.nan_to_num(mat)
    sm = ndi.gaussian_filter1d(mat, 1, axis=1, mode="nearest")
    avg = sm.mean(0)
    if avg.max() <= 0:
        return []
    above = avg > 0.2 * avg.max()
    thr = mat.mean() + mat.std()
    out = []
    i = 0
    while i < len(above):
        if above[i]:
            j = i
            while j + 1 < len(above) and above[j + 1]:
                j += 1
            if j - i + 1 >= 2 and np.sum((mat[:, i:j + 1] > thr).any(1)) >= 8:
                out.append((i, j + 1))
            i = j + 1
        else:
            i += 1
    return out
def com(v, lo, hi):
    x = np.arange(lo, hi); w = np.clip(np.nan_to_num(v[lo:hi]), 0, None)
    return (x * w).sum() / w.sum() if w.sum() > 0 else np.nan
FLD = []
for k in CL:
    r = res[k]; pc_idx = r["pc_idx"]; X = r["tm_dec_pc"].astype(float)
    s0 = np.asarray(r["sets"][0])[:30]; s1 = np.asarray(r["sets"][1]); s1l = s1[-30:]
    pc0, pc1 = r["sig"]
    back = r["zone_starts"][1] < r["zone_starts"][0]
    for j, ci in enumerate(pc_idx):
        cat = next((c for c in ("rr", "tr", "nonrr", "app") if CL[k][c][ci]), None)
        if cat is None:
            continue
        f0 = fields_of(X[s0, :, j]); f1 = fields_of(X[s1l, :, j])
        inc = (pc0[ci] or len(f0) > 0) and (pc1[ci] or len(f1) > 0)
        row = dict(mouse=k[0], day=k[1], cell=ci, cat=cat, n0=len(f0), n1=len(f1), included=inc, backward=back)
        if len(f0) == 2 and len(f1) == 2:
            c0 = [com(X[s0, :, j].mean(0), a, b) for a, b in f0]; c1 = [com(X[s1l, :, j].mean(0), a, b) for a, b in f1]
            row["off0"] = (c0[1] - c0[0]) * 2 * np.pi / 45; row["off1"] = (c1[1] - c1[0]) * 2 * np.pi / 45
        if len(f1):
            mpost = X[s1l, :, j].mean(0)
            a, b = max(f1, key=lambda ab: np.nanmax(mpost[ab[0]:ab[1]]))      # primary field: the one with the peak
            lo, hi = max(0, a - 1), min(45, b + 1)                            # +/- 10 cm
            infield = np.nan_to_num(X[s1, lo:hi, j])
            allpost = np.nan_to_num(X[s1, :, j])
            on = infield.max(1) > allpost.mean() + allpost.std()   # mean + 1 s.d. of the post-switch binned activity
            form = np.nan
            for t in range(len(on) - 4):
                if on[t:t + 5].sum() >= 3:
                    form = t + int(np.argmax(on[t:t + 5])); break
            if np.isfinite(form):
                fl = int(form)
                row["form_lap"] = fl + 1                                       # lap after the switch, from 1
                row["shift_cm"] = (com(mpost, lo, hi) - com(np.nan_to_num(X[s1[fl], :, j]), lo, hi)) * 10
        FLD.append(row)
FLD = pd.DataFrame(FLD)
print("cells in RR, TR, non-RR or appearing with per-trial maps:", len(FLD), "; included (SI or field before and after):", int(FLD.included.sum()))

## ed3d-single-field-increases

> The fraction of cells keeping a single place field both before and after the switch increased across switch days for all subpopulations, most clearly for RR cells.

In [ ]:
for cat in ("rr", "tr", "nonrr"):
    F = FLD[(FLD.cat == cat) & FLD.included]
    by = F.groupby("day").apply(lambda g: np.mean((g.n0 == 1) & (g.n1 == 1)))
    lr = st.linregress(np.arange(1, 8), by.loc[sosa.SWITCH_DAYS].values)
    print(f"{cat:6s}: fraction with one field before and after by day {by.round(3).tolist()}; slope {lr.slope:+.4f}, p = {lr.pvalue:.3g}, n cells {len(F)}")

**Result: not reproduced.** The fraction with one field before and after does not rise for any subpopulation (slopes -0.0010, -0.0067, -0.0014; p 0.81, 0.24, 0.69).

## ed3fg-fields-remap-independently

> For RR cells with two fields before and after the switch, the offsets between the two fields were not correlated before against after, indicating independent remapping of the fields.

In [ ]:
F = FLD[(FLD.cat == "rr") & FLD.included & FLD.off0.notna()]
for d in sosa.SWITCH_DAYS:
    g = F[F.day == d]
    if len(g) >= 3:
        rho = sosa.corr_cc(g.off0.values, g.off1.values)
        # two-sided p from the circular-circular correlation statistic (Jammalamadaka & SenGupta / Kempter)
        a, b = g.off0.values, g.off1.values
        ma, mb = sosa.circmean(a), sosa.circmean(b)
        l20 = np.mean(np.sin(a - ma) ** 2); l02 = np.mean(np.sin(b - mb) ** 2); l22 = np.mean(np.sin(a - ma) ** 2 * np.sin(b - mb) ** 2)
        z = np.sqrt(len(a) * l20 * l02 / l22) * rho
        p = 2 * (1 - st.norm.cdf(abs(z)))
        print(f"day {d:2d}: n = {len(g)}, rho = {rho:.3f}, p = {p:.3g}")
    else:
        print(f"day {d:2d}: n = {len(g)}, too few cells")
print("expected n: day 3 31 +/- 3.1, day 8 20 +/- 2")

**Result: not reproduced.** Two of seven days show a significant correlation of field offsets (day 5 rho = 0.267, p = 0.0068; day 14 rho = -0.187, p = 0.036). The cells with two fields before and after (103 on day 3, 153 on day 8) are three to eight times the paper's 31 and 20, so the field detection here differs from the paper's.

## ed3mo-appearing-late-backward

> Appearing cells formed their fields on later laps and shifted backward more after the formation lap than RR, TR and non-RR remapping cells.

In [ ]:
F = FLD[FLD.form_lap.notna() & (FLD.included | (FLD.cat == "app"))]
print("primary fields with a formation lap:", F.groupby("cat").size().to_dict(), "(expected RR 2368, appearing 2386)")
print("median formation lap:", F.groupby("cat").form_lap.median().to_dict(), "; median shift (cm):", F.groupby("cat").shift_cm.median().round(1).to_dict())
ok = True
for c in ("rr", "tr", "nonrr"):
    z1, p1 = st.ranksums(F[F.cat == c].form_lap, F[F.cat == "app"].form_lap)
    z2, p2 = st.ranksums(F[F.cat == c].shift_cm, F[F.cat == "app"].shift_cm)
    ok &= (z1 < 0 and p1 < 0.0083 and z2 > 0 and p2 < 0.004)
    print(f"{c:6s} vs appearing: formation lap Z = {z1:.1f}, p = {p1:.2g}; shift Z = {z2:.1f}, p = {p2:.2g}")
print("appearing later and more backward than each, at the legend's levels:", ok)

**Result: reproduced.** Appearing fields form later (median lap 6 against 1 to 3) and shift more backward (median -6.1 cm against -0.5 to -3.2) than each other subpopulation, at the legend's levels (largest p 0.00011). Primary fields: appearing 2,140 (within tolerance of 2,386), RR 6,328 (the paper's 2,368).

## ed3p-shift-follows-reward-direction

> Primary fields shifted more backward when the reward moved backward on the track than when it moved forward, in every subpopulation.

In [ ]:
F = FLD[FLD.form_lap.notna() & (FLD.included | (FLD.cat == "app"))].copy()
F["sess"] = F.mouse + "_" + F.day.astype(str)
rng = np.random.default_rng(3001)
sess_dir = F.groupby("sess").backward.first()
for c in ("rr", "tr", "nonrr", "app"):
    g = F[F.cat == c]
    zb, _ = st.ranksums(g[g.backward].shift_cm, g[~g.backward].shift_cm)
    null = []
    for _ in range(100):
        perm = pd.Series(rng.permutation(sess_dir.values), index=sess_dir.index)
        lab = g.sess.map(perm).values.astype(bool)
        null.append(st.ranksums(g.shift_cm[lab], g.shift_cm[~lab])[0])
    p = (np.sum(np.abs(null) >= abs(zb)) + 1) / 101
    print(f"{c:6s}: backward n {int(g.backward.sum())}, median {g[g.backward].shift_cm.median():.1f} cm; forward n {int((~g.backward).sum())}, "
          f"median {g[~g.backward].shift_cm.median():.1f} cm; Z = {zb:.1f}, permutation p = {p:.4f} (minimum 1/101 = 0.0099)")

**Result: reproduced.** In every subpopulation, fields shift more backward when the reward moves backward (medians -5.5 against 0.6 cm for RR, and similar for the others), and the session-label permutation p is 0.0099 for each, the smallest 100 permutations can give.

## Blocked claims

- fig3b-decode-rr-position, fig3c-decode-range (too large). The decoder needs each session's activity time series, which this notebook reduces to binned maps as it loads, and its own pass over the files (about 30 minutes). The paper's procedure takes 77 sessions x 3 populations x (1 + 100 shuffles) x 10 folds, about 233,000 fits of the circular regression (decode.py, up to 100 EM iterations each), before the tests after the switch. At the ~20 ms a fit of this size takes, that is more than an hour of fitting. This is more than the time left in this run and more than the notebook's limit together with the rest of the analysis.
- glm-well-fit-cells, fig7de-glm-top-predictors (too large). The GLM (GLM_Tensorflow_2, group lasso over 11 values of lambda, fivefold cross-validation, ablation per variable) is fit to every place cell (35,386 in the paper, about 44,000 here) on its time series. It needs TensorFlow and many hours of fitting.
- fig8e-rr-remap-precedes-behavior, ed10-appearing-remap-later (other). The factorized k-means with rotation-shuffle model selection (lvl) and the sigmoid fits were not attempted in this run's time.
- ed5-teleport-fractions (other). It needs place cells re-identified on bins that extend into the teleport, from the time series of the sessions imaged through it. This notebook's per-session reduction does not keep those, and the analysis was not attempted in this run's time. The data check shows which sessions were imaged through the teleport (no dark frames). They are the sessions the Methods name: m11 to m14 on days 1, 7, 8 and 14, and m15 to m19 on day 1 and the switch days.

## Summary

| Claim | Outcome |
|---|---|
| fig1h-lick-ratio-improves | reproduced (reading b) |
| place-cell-fraction | partially reproduced |
| remap-category-fractions | partially reproduced |
| ed2cd-switch-remapping-intermediate | reproduced |
| ed2f-teleport-distance | reproduced |
| fig2e-rr-remapping-above-chance | reproduced |
| fig2h-rr-remapping-grows | reproduced |
| fig2eg-mean-rr-position-after-reward | reproduced |
| fig2i-rr-remapping-spans-track | reproduced |
| ed2jl-rr-remapping-beyond-50cm | partially reproduced |
| ed2m-more-fields-after-reward | partially reproduced |
| ed2n-far-remapping-no-increase | reproduced |
| rr-cell-fractions | partially reproduced |
| ed2pq-rr-reward-aligned-correlation | reproduced |
| fig3b-decode-rr-position | blocked, too large |
| fig3c-decode-range | blocked, too large |
| fig4c-rr-sequences-preserved | reproduced |
| fig4f-tr-sequences-preserved | reproduced |
| fig4i-nonrr-sequences-not-preserved | reproduced |
| ed5a-tr-fewer-across-environments | partially reproduced |
| ed5f-rr-no-decrease-across-environments | reproduced |
| fig4j-rr-overrepresent-reward | partially reproduced |
| fig4m-rr-density-near-reward-grows | reproduced |
| fig4no-licking-speed-increase | reproduced |
| fig4p-lick-precision-sequence-precision | reproduced (reading a) |
| ed4hij-sequence-variance-decreases | partially reproduced |
| fig4qr-tr-track-ends | reproduced |
| ed4l-disappearing-decrease | partially reproduced |
| ed5-teleport-fractions | blocked, other |
| fig5b-rr-remain-rr-increases | reproduced |
| fig5d-nontr-to-rr-increases | not reproduced |
| fig5f-nonpc-to-rr-increases | not reproduced |
| fig5h-tr-to-rr-constant | reproduced |
| fig5i-rr-to-tr-decreases | reproduced |
| fig5jkl-tr-no-recruitment | reproduced |
| ed6a-rr-near-reward-stable | reproduced |
| ed6c-rr-shift-toward-reward | not reproduced |
| ed6-across-day-drift | reproduced |
| ed7-no-anatomical-bias | partially reproduced |
| fig6e-rr-prefer-reward | partially reproduced |
| ed8k-rr-prefer-reward-after-switch | reproduced |
| ed8df-time-warp-speed-mse | reproduced |
| glm-well-fit-cells | blocked, too large |
| fig7de-glm-top-predictors | blocked, too large |
| fig8e-rr-remap-precedes-behavior | blocked, other |
| ed10-appearing-remap-later | blocked, other |
| ed3d-single-field-increases | not reproduced |
| ed3fg-fields-remap-independently | not reproduced |
| ed3mo-appearing-late-backward | reproduced |
| ed3p-shift-follows-reward-direction | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's.
- The dataset version is 0.251124.0550, not the cited 0.250406.0045, whose behaviour is one session's copy per mouse.
- dF/F and the deconvolved activity are recomputed from F and the neuropil with the code's parameters (neuropil coefficient 0.7, maximin over 300 samples after a 15-sample Gaussian, OASIS with tau 0.7 s per trial). The paper's own traces are not in the files.
- More place cells pass the spatial-information test here (60% against 48.5%), so pooled counts of cells (RR, TR, non-RR, remapping cells) are 13 to 50% above the paper's, while many fractions and every test of direction agree.
- Cells are matched across days here (the files hold no matching), by the IoU rule of the Methods after a rigid alignment of the mean images. This is an assumption.
- Omission trials are taken as unrewarded trials with licks in the zone. This is an assumption, and the per-mouse day counts it gives equal the paper's.
- Place fields and formation laps use 10 cm binned events, not the raw events. The two-field counts differ strongly from the paper's.

What was found about the dataset.
- In the cited version every file of a mouse holds the same session's behaviour. This is fixed in 0.251124.0550.
- The files' "Deconvolved" series (unit "lumens", no description) is not the paper's deconvolved activity: its median correlation with OASIS on dF/F is 0.47.
- The files hold no cross-day ROI matching, no depth estimates, and no omission labels, and they hold none of the three fixed-condition mice.
- The pixel masks' "x" field indexes image rows.
- The `reward_zone` series is described as binary but takes values 0 to 6, and it does not rise on unrewarded trials.
- The scene names, the number of trials, the lick-sensor errors (81 trials) and the sessions imaged through the teleport agree with the paper.